# Wczytywanie danych do ramki danych pandas

In [98]:
import pandas as pd
from pathlib import Path

# Względna ścieżka do pliku JSON z ofertami pracy
input_path = Path("../scrapping-worker/justjoinit/offers.json")

# Wczytanie danych do ramki danych pandas (na tym etapie dane są "nieczyste")
df: pd.DataFrame = pd.read_json(
    input_path,
    orient="records"
)

# Podgląd pierwszych wierszy DataFrame
df.head()

,Details,Tech stack,Salary
0,{},{},Undisclosed salary
1,{},{},Undisclosed salary
2,{},{},Undisclosed salary
3,"{'Type of work': 'Freelance', 'Experience': 'S...","{'AWS': 'master', 'ETL': 'advanced', 'Power BI...",Undisclosed salary
4,{},{},Undisclosed salary


# Rozwijanie zagnieżdzionch słowników

### W niektórych kolumnach, takich jak "Details" czy "Tech Stack", mogą znajdować się zagnieżdżone słowniki 
### Aby uzyskać z nich dodatkowe atrybuty jako osobne kolumny w DataFrame, można je „rozwinąć” za pomocą pd.apply(pd.Series)


In [99]:
# UWAGA:
# Jeśli uruchomisz tę komórkę wielokrotnie bez ponownego wczytania danych,
# zmienna 'df' nie będzie już zawierać kolumn z listy 'columns_to_flatten'.
# W takim przypadku należy ponownie uruchomić komórkę wczytującą dane z pliku .json.

def flatten_columns(df: pd.DataFrame, col_label: str) -> pd.DataFrame:
    """
    Rozwija słowniki zawarte w kolumnie `col_label` ramki danych `df`
    i zwraca nowy DataFrame z rozwiniętymi kolumnami.
    """
    expanded_df = df[col_label].apply(pd.Series)
    expanded_df.fillna("Not included", inplace=True)
    df.drop(labels=col_label, axis=1, inplace=True)
    return expanded_df

def flatten_multiple_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Rozwija wiele kolumn zawierających słowniki i łączy je z oryginalnym DataFrame.
    """
    flattened_dataframes = []
    for col_label in columns:
        flatten_df = flatten_columns(df, col_label)
        flattened_dataframes.append(flatten_df)

    return pd.concat([df] + flattened_dataframes, axis=1)

# Lista kolumn do rozwinięcia
columns_to_flatten: list[str] = ["Details", "Tech stack"]

# Rozwijanie wskazanych kolumn i łączenie z oryginalnym DataFrame
offers_df: pd.DataFrame = flatten_multiple_columns(df, columns_to_flatten)

# Pobierz listę kolumn z DataFrame
offers_cols: list[str] = offers_df.columns.to_list()

# Popraw nazwę kolumny, jeśli występuje literówka
if ", Prometheus" in offers_cols:
    offers_cols[offers_cols.index(", Prometheus")] = "Prometheus"

# Zaaktualizuj nazwy kolumn
offers_df.columns = offers_cols

In [100]:
offers_df.head()

,Salary,Type of work,Experience,Employment Type,Operating mode,AWS,ETL,Power BI,Python,MySQL,Ansible,Prometheus,PMM,Linux server systems
0,Undisclosed salary,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included
1,Undisclosed salary,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included
2,Undisclosed salary,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included
3,Undisclosed salary,Freelance,Senior,B2B,Hybrid,master,advanced,advanced,advanced,Not included,Not included,Not included,Not included,Not included
4,Undisclosed salary,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included
